In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# 04 - Unsupervised anomaly detection
Isolation Forest and Local Outlier Factor trained **without labels**. Useful when fraud labels are delayed or novel attacks appear.

In [2]:
from src.anomaly_detection.detectors import fit_isolation_forest, iso_scores, fit_lof, lof_scores
from sklearn.metrics import roc_auc_score, average_precision_score
fe = load_features(); d = split_xy(fe); X_tr, y_tr, _ = d['train']; X_te, y_te, _ = d['test']

In [3]:
iso = fit_isolation_forest(X_tr, 0.02, CFG['seed']); si = iso_scores(iso, X_te)
lof = fit_lof(X_tr, 6000, 35, 0.02, CFG['seed']); sl = lof_scores(lof, X_te)
pd.DataFrame({'ROC-AUC': {'isolation_forest': roc_auc_score(y_te, si), 'lof': roc_auc_score(y_te, sl)}, 'PR-AUC': {'isolation_forest': average_precision_score(y_te, si), 'lof': average_precision_score(y_te, sl)}}).round(3)

,ROC-AUC,PR-AUC
isolation_forest,0.780,0.400
lof,0.683,0.116


In [4]:
# Precision when we review only the top-k most anomalous transactions
rows = []
for k in (50, 100, 200, 500):
    for name, s in (('iso', si), ('lof', sl)):
        top = np.argsort(-s)[:k]; rows.append({'k': k, 'detector': name, 'precision@k': y_te[top].mean(), 'recall@k': y_te[top].sum() / y_te.sum()})
pd.DataFrame(rows).pivot(index='k', columns='detector').round(3)

precision@k       recall@k       
detector         iso   lof      iso    lof
k                                         
50             0.860  0.26    0.247  0.075
100            0.560  0.26    0.322  0.149
200            0.380  0.24    0.437  0.276
500            0.178  0.13    0.511  0.374

In [5]:
# sensitivity to contamination
from src.model_evaluation.evaluate import evaluate_binary
out = []
for c in (0.005, 0.01, 0.02, 0.05, 0.1):
    thr = np.quantile(si, 1-c); r = evaluate_binary(y_te, si, thr); out.append({'contamination': c, 'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'], 'FPR': r['false_positive_rate']})
pd.DataFrame(out).round(3)

,contamination,precision,recall,f1,FPR
0,0.005,0.889,0.230,0.365,0.001
1,0.010,0.622,0.322,0.424,0.004
2,0.020,0.411,0.425,0.418,0.012
3,0.050,0.196,0.506,0.283,0.041
4,0.100,0.106,0.546,0.178,0.091


In [6]:
sns.kdeplot(x=si, hue=y_te, common_norm=False); plt.title('Isolation Forest score: legit (0) vs fraud (1)')

Text(0.5, 1.0, 'Isolation Forest score: legit (0) vs fraud (1)')

Unsupervised scores rank fraud far above chance but trail supervised models - the price of not using labels. Combining both (score as an extra feature) is listed under Future Work.